# Efficient Thumbnail Caching

This notebook adds photos from a specific country to the local JSON cache using **concurrent workers**. This is much faster than sequentially fetching and processing each thumbnail.

In [1]:
import os
import sys
import concurrent.futures
from pathlib import Path
from tqdm.auto import tqdm

# Ensure we can import from the project root
sys.path.append(str(Path.cwd().parent))

from src.config import IMMICH_URL, API_KEY, CACHE_PATH, MAX_WORKERS
from src.immich_client import ImmichClient
from src.analyzer import analyze_thumbnail
from src.cache import ThumbnailCache
from src.data_processor import load_and_preprocess_photos

c:\Users\Aya\Desktop\Immich\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print("Loading Data...")
df, travel, home = load_and_preprocess_photos()

print("Loading Cache...")
cache = ThumbnailCache(CACHE_PATH)

Loading Data...
Loading Cache...
[cache] Loaded 2951 entries from C:\Users\Aya\Desktop\Immich\data\thumbnails_analysis.json


In [3]:
countries = travel["country"].unique()
print(countries)

<StringArray>
[               'Saudi Arabia',        'United Arab Emirates',
                   'Sri Lanka',                    'Malaysia',
                    'Cambodia',                     'Vietnam',
                   'Indonesia',                       'Qatar',
                       'Kenya', 'United Republic of Tanzania',
                       'Japan',  'People's Republic of China']
Length: 12, dtype: str


In [11]:
TARGET_COUNTRY = "People's Republic of China"  # Change this to any country in your dataset!

# Get all asset IDs for this country
country_ids = travel[travel["country"] == TARGET_COUNTRY]["id"].tolist()
print(f"Found {len(country_ids)} photos for {TARGET_COUNTRY}")

# Filter out what we already have
missing_ids = cache.missing_from(country_ids)
print(f"Need to download/analyze {len(missing_ids)} missing thumbnails.")

Found 500 photos for People's Republic of China
Need to download/analyze 0 missing thumbnails.


In [9]:
client = ImmichClient(base_url=IMMICH_URL, api_key=API_KEY)

def process_asset(asset_id: str):
    """Downloads thumbnail and analyzes it in one go. Returns (asset_id, result_dict, err_msg)."""
    try:
        img = client.get_thumbnail(asset_id, size="preview")
        if img:
            result = analyze_thumbnail(img)
            return asset_id, result, None
        return asset_id, None, "Could not fetch thumbnail"
    except Exception as e:
        return asset_id, None, str(e)

if missing_ids:
    print(f"Starting {MAX_WORKERS} concurrent workers to process {len(missing_ids)} thumbnails...")
    
    # Provide concurrent execution using threads
    with concurrent.futures.ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        future_to_id = {executor.submit(process_asset, aid): aid for aid in missing_ids}
        
        # Use tqdm to track progress as futures complete
        for i, future in enumerate(tqdm(concurrent.futures.as_completed(future_to_id), total=len(missing_ids), desc="Caching")):
            asset_id, result, err = future.result()
            
            if result:
                cache[asset_id] = result  # Add to in-memory dict
            elif err:
                tqdm.write(f"[ERROR] Failed to process {asset_id}: {err}")
                
            # Let cache handle saving checkpoint internally
            cache.checkpoint(i, every=200, total=len(missing_ids))

    # Important: Final save ensures last batch is persisted to disk
    cache.save()
    print("Caching complete!")
else:
    print(f"All {TARGET_COUNTRY} photos are already fully cached!")

Starting 8 concurrent workers to process 2 thumbnails...


Caching: 100%|██████████| 2/2 [00:00<00:00, 13.21it/s]


Caching complete!
